# Intro to Scaled Qualitative Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Corey-Abramson/Intro-to-Scaled-Qualitative-Analysis/blob/main/notebook/intro_scaled_qualitative_analysis.ipynb)

Interview transcripts arrive as text. Analysing them at scale needs rows. This
notebook runs that conversion end to end on public interview data, then codes
the result three different ways and draws two figures from the codes it made.

**normalize → read into a table → classify → visualize**

| Stage | What you will do |
|---|---|
| **1. Normalize** | Turn one messy raw transcript into machine-readable rows |
| **2. Read into a table** | Load a coded corpus that already uses that format |
| **3. Classify** | Code text three ways: a dictionary, a model, and a language model |
| **4. Visualize** | Draw a word cloud and a code co-occurrence heatmap |

Run the cells top to bottom. It works locally and in Colab. There are no API
keys, no model downloads, and nothing to configure.

**Licensing.** The code here is BSD 3-Clause. The dataset is **not**. It keeps
the IEEE History Center's own restriction on quotation. See
[`SAMPLE_DATA.md`](../SAMPLE_DATA.md).

**References.** Rather than a wall of citations here, see the lab's curated
[topical bibliography](https://github.com/Computational-Ethnography-Lab/teaching#v-bibliography)
and the [Computational Ethnography Lab](https://computationalethnography.org/).
Works cited in this notebook are listed in [`REFERENCES.md`](../REFERENCES.md).

In [ ]:
# Setup. Finds the repository, installs what is missing, checks versions.

import os
import subprocess
import sys
from pathlib import Path

RELEASE_TAG = "v1.0.0"
REPO_URL = "https://github.com/Corey-Abramson/Intro-to-Scaled-Qualitative-Analysis.git"
SENTINEL = Path("cmap_demo") / "__init__.py"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo_root(start):
    """Walk up from `start` looking for the sentinel file."""
    for candidate in [start, *start.parents]:
        if (candidate / SENTINEL).exists():
            return candidate
    return None


root = find_repo_root(Path.cwd().resolve())

if root is not None:
    # Branch A: the repository is already on disk. Nothing to fetch.
    os.chdir(root)
    print(f"[OK] Found the repository at {root}")
elif IN_COLAB:
    # Branch B: opened from the Colab badge, which brings only this notebook.
    # Clone the pinned release next to it.
    target = Path("/content") / f"Intro-to-Scaled-Qualitative-Analysis-{RELEASE_TAG}"
    if not (target / SENTINEL).exists():
        print(f"[..] Cloning {RELEASE_TAG} ...")
        subprocess.run(
            ["git", "clone", "--branch", RELEASE_TAG, "--depth", "1",
             REPO_URL, str(target)],
            check=True,
        )
    os.chdir(target)
    checked_out = subprocess.run(
        ["git", "describe", "--tags", "--exact-match"],
        capture_output=True, text=True,
    ).stdout.strip()
    if checked_out != RELEASE_TAG:
        raise RuntimeError(
            f"Expected release {RELEASE_TAG} but the checkout reports "
            f"{checked_out!r}. Stopping rather than running unknown code."
        )
    root = target
    print(f"[OK] Cloned {RELEASE_TAG} to {root}")
else:
    raise RuntimeError(
        "Could not find cmap_demo/__init__.py by walking up from "
        f"{Path.cwd()}. Run this notebook from inside a clone of the "
        "repository, or open it with the Colab badge."
    )

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Install only what is missing. On Colab the preinstalled numpy/pandas/scipy
# are left alone: forcing a downgrade there is slow and breaks other things.
REQUIRED = {
    "pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "scipy": "scipy", "wordcloud": "wordcloud",
    "PIL": "pillow",
}
missing = []
for module_name, package_name in REQUIRED.items():
    try:
        __import__(module_name)
    except ImportError:
        missing.append(package_name)

if missing:
    print(f"[..] Installing {', '.join(missing)} ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing],
        check=True,
    )
    print("[OK] Installed")
else:
    print("[OK] All required packages already present")

# Version gate. These bounds were verified by running this whole workflow on
# numpy 1.26.4 / pandas 2.1.4, numpy 2.2.6 / pandas 2.2.3, and
# numpy 2.4.6 / pandas 3.0.5. Outside them, stop rather than fail mid-figure.
import numpy as np
import pandas as pd


def _major_minor(version):
    parts = version.split(".")
    return int(parts[0]), int(parts[1])


for name, version, low, high in [
    ("numpy", np.__version__, (1, 26), (3, 0)),
    ("pandas", pd.__version__, (2, 1), (4, 0)),
]:
    if not (low <= _major_minor(version) < high):
        raise RuntimeError(
            f"{name} {version} is outside the tested range "
            f"{low[0]}.{low[1]} to below {high[0]}.{high[1]}. "
            f"Install a supported version with: pip install -r requirements.txt"
        )
print(f"[OK] numpy {np.__version__}, pandas {pd.__version__} are in the tested range")

# Figures and CSVs land here. Git does not keep an empty ignored directory, so
# a fresh clone has no output/ until this runs.
OUTPUT_DIR = Path("output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# NLTK is optional. If these downloads fail or are blocked, the word cloud
# falls back to a bundled stop-word list and a regex tokenizer.
try:
    import nltk
    for resource in ["punkt", "punkt_tab", "stopwords"]:
        try:
            nltk.download(resource, quiet=True)
        except Exception:
            pass
    print("[OK] NLTK data requested (optional)")
except ImportError:
    print("[--] NLTK not installed; the bundled fallback will be used")

from cmap_demo.header import print_header

print()
print_header(stage="Setup complete")

## Stage 1. Making text machine-readable

Three things are going on when a transcript becomes a table.

**One row per unit of talk, with stable identifiers.** The unit here is a
speaker turn. Each row carries a `doc_id` that does not change, so a code
attached to row 47 stays attached to row 47 through every later step.

**The transformation is reconstructible.** Each row also records where its text
started and ended in the normalized document. The rows are not a summary of the
transcript; the normalized transcript can be rebuilt from them. Cell 5 does
exactly that and checks the result character for character.

**The format is a fixed schema.** Twelve columns, the same ones the coded
corpus in Stage 2 uses, which is why rows this notebook produces and rows read
from that corpus are interchangeable.

| Column | Holds |
|---|---|
| `project` | Which body of data this row belongs to |
| `number` | The segment label; here, the speaker of the turn |
| `reference` | Position of the turn within the document |
| `text` | The normalized text of the turn |
| `document` | Which transcript the row came from |
| `start_position`, `end_position` | Character offsets in the normalized document |
| `data_group` | The kind of data, as a list: `['interview']` |
| `text_length`, `word_count` | Size of the text |
| `doc_id` | Stable unique row identifier |
| `codes` | The codes on this row, as a list |

Representing qualitative material as arrays so it can be analysed
computationally, while keeping it tied back to the original, is developed in
Abramson and Dohan (2015) and Abramson et al. (2018).

In [ ]:
# A raw transcript, exactly as it arrives. This one is synthetic, written for
# this repository so there is something messy to clean.

RAW_PATH = Path("data/raw/interview_demo01_20240115.txt")
raw_text = RAW_PATH.read_text()

print(f"{RAW_PATH}  ({len(raw_text):,} characters)")
print("-" * 72)
for line in raw_text.splitlines()[:15]:
    print(repr(line))

In [ ]:
# Filename to metadata, then clean the text.

from cmap_demo.normalize import (
    collapse_whitespace, detect_speaker, parse_filename, remove_timestamps,
)

metadata = parse_filename(RAW_PATH.name)
print("Filename convention: datasource_subject_date.txt")
print(f"  {RAW_PATH.name} -> {metadata}")
print()

# A non-conforming name fails loudly rather than guessing.
try:
    parse_filename("notes.txt")
except ValueError as error:
    print(f"Rejected as expected: {error}")
print()

print("Before and after, on the first three lines that carry a turn:")
print("-" * 72)
shown = 0
for line in raw_text.splitlines():
    if shown >= 3 or not line.strip():
        continue
    if remove_timestamps(line) == line and detect_speaker(line)[0] is None:
        continue  # front matter, nothing for this step to do
    cleaned = collapse_whitespace(remove_timestamps(line))
    speaker, remainder = detect_speaker(cleaned)
    print(f"  before : {line!r}")
    print(f"  after  : {cleaned!r}")
    print(f"  speaker: {speaker!r}")
    print(f"  text   : {remainder!r}")
    print()
    shown += 1

In [ ]:
# One row per speaker turn, in the CMAP schema, then validate it.

from cmap_demo.normalize import (
    build_cmap_frame, reconstruct_text, segment_turns, validate_cmap_frame,
)

turns = segment_turns(raw_text)
normalized = build_cmap_frame(turns, metadata)

normalized_path = OUTPUT_DIR / "normalized_sample.csv"
normalized.to_csv(normalized_path, index=False)
print(f"{len(turns)} turns -> {normalized_path}")

# Validation is not a column-name check. It confirms the required fields are
# non-empty, that codes and data_group round-trip to lists, that doc_id is
# unique, that word_count matches the text, and that the file re-reads to an
# identical frame.
validate_cmap_frame(normalized, csv_path=normalized_path)

# The reconstructibility claim from Stage 1, checked rather than asserted.
rebuilt = reconstruct_text(normalized)
expected = " ".join(normalized["text"])
print(f"Rebuilt {len(rebuilt):,} characters from the rows and their offsets; "
      f"identical to the normalized document: {rebuilt == expected}")
print()

normalized[["doc_id", "number", "word_count", "text"]].head()

## De-identification: where it goes, and why it is skipped here

In the production pipeline the next stage is de-identification: names, places,
employers, and dates are detected and replaced before the text goes any
further. It sits here, immediately after normalization, because everything
downstream inherits whatever it lets through.

It is **required** for human-subjects data, and no amount of care later
compensates for skipping it.

This notebook does not run it. The demo corpus is a published, public set of
oral histories, so there is nothing to remove, and the one raw transcript
above is synthetic and describes no real person. Skipping it here is a property
of this particular data, not a shortcut you should copy.

## Stage 2. A coded corpus in the same format

`data/1_cleaned_data.csv` holds engineering oral histories in the twelve-column
format Stage 1 just produced. The provenance chain runs:

**IEEE History Center**, the Engineering and Technology History Wiki, where
these oral histories were published → **ASA 2022 workshop**, Zhuofan Li's
workshop materials, which brought the corpus into a coded, tabular form →
**CMAP format**, the shape used here and in the CMAP Visualization Toolkit.

Because it shares the schema, everything below would work the same way on rows
this notebook made itself.

> The data carries its own restriction and is **not** covered by this
> repository's BSD licence. No part of it may be quoted for publication
> without written permission from the Director of the IEEE History Center.
> See [`SAMPLE_DATA.md`](../SAMPLE_DATA.md).

In [ ]:
# Load the coded corpus and take a workable subset.

corpus = pd.read_csv("data/1_cleaned_data.csv")

print(f"rows      : {len(corpus):,}")
print(f"documents : {corpus['document'].nunique()}")
print(f"columns   : {len(corpus.columns)}")
print(f"            {list(corpus.columns)}")
print()

# Most rows are very short: the median turn is 16 words, and plenty are a
# single word. Keep rows with something to analyse, from the first 40
# documents, so the figures below are quick.
demo = corpus[corpus["word_count"] >= 25]
first_documents = demo["document"].unique()[:40]
demo = demo[demo["document"].isin(first_documents)].copy()

print(f"demo subset: {len(demo):,} rows across {demo['document'].nunique()} documents")
demo[["doc_id", "document", "word_count", "text"]].head(3)

## Stage 3. Three ways to code text at scale

**A dictionary.** You write the patterns; the machine applies them. Completely
transparent and completely reproducible, and it only ever finds what you
thought to look for. Always read the matches; a term that looks obvious often
fires on something you did not intend.

**A model.** Segments are embedded and a classifier is trained on labelled
examples. Handles phrasing a dictionary would miss, needs labelled data, and
what it learned is much harder to inspect.

**A large language model.** Instructions in prose instead of patterns or
training data. Fast to start and easy to get wrong: outputs drift between runs,
and a model will happily produce confident output about text it never read.

They are not rivals. The dictionary gives you a transparent floor, the model
scales it, and the language model handles what neither anticipated, and the
hybrid human-machine combination is where the useful work tends to happen
(Li, Dohan and Abramson 2021).

In [ ]:
# Lane 1. Dictionary and regex, with real matches on the real corpus.

from collections import Counter

from cmap_demo.llm_handoff import (
    ALLOWED_CODES, CONCEPT_DICTIONARIES, code_by_dictionary, dictionary_matches,
)

print("Concept dictionaries:")
for code_name, pattern in CONCEPT_DICTIONARIES.items():
    print(f"  {code_name:<10} {pattern.pattern[:64]}...")
print()

dictionary_codes = [code_by_dictionary(text) for text in demo["text"]]
counts = Counter(code for row in dictionary_codes for code in row)

print(f"Hits across {len(demo):,} rows:")
for code_name in ALLOWED_CODES:
    print(f"  {code_name:<10} {counts[code_name]:>5} rows")
print()

# A dictionary that matches nothing is a broken dictionary, not an empty corpus.
empty = [c for c in ALLOWED_CODES if counts[c] == 0]
if empty:
    raise RuntimeError(f"These dictionaries matched nothing: {empty}. Fix them.")

# Read the actual matches before trusting any of the counts above.
print("Actual matched text:")
print("-" * 72)
shown = 0
for _, row in demo.iterrows():
    hits = dictionary_matches(row["text"])
    if not hits or shown >= 5:
        continue
    excerpt = " ".join(str(row["text"]).split())[:150]
    print(f"  doc_id {row['doc_id']} -> {sorted({c for c, _ in hits})}")
    print(f"    matched: {sorted({w.lower() for _, w in hits})}")
    print(f"    text   : {excerpt}...")
    print()
    shown += 1

## Lane 2. The model lane

This is the lane that needs a transformer, labelled examples, and a training
step, so this notebook links it rather than shipping a model download that
would stall a live demo.

- **[Zhuofan Li's ASA 2022 workshop](https://github.com/lizhuofan95/ASA2022_Workshop)**
  gives the full transformer walkthrough on this same corpus, with a runnable
  [Colab notebook](https://colab.research.google.com/drive/1qMwvjaY6DKQ-jxFTyXt3S3qNQdpV_S9n)
- **[CMAP Visualization Toolkit](https://github.com/Computational-Ethnography-Lab/cmap_visualization_toolkit)**
  covers embeddings, clustering, t-SNE, and semantic networks over coded corpora
- **[Using machine learning with ethnographic interviews](https://cmabramson.com/resources/f/using-machine-learning-with-ethnographic-interviews)**
  is the author's methods-resources page

The result behind this lane is that human and machine coding combined
outperform either alone, reported in Li, Dohan and Abramson (2021),
"Qualitative Coding in the Computational Era," *Socius* 7
([doi:10.1177/23780231211062345](https://doi.org/10.1177/23780231211062345)).
That finding is theirs and is cited here, not recomputed below.

The figures in the next cell come from the CMAP Visualization Toolkit and show
what this class of output looks like at full scale.

In [ ]:
# Two figures from the CMAP Visualization Toolkit (BSD 3-Clause), showing what
# the full toolkit produces. These are the toolkit's own published figures,
# copied byte for byte from its documentation, not generated here.

from IPython.display import Image, display

for caption, path in [
    ("Code co-occurrence heatmap, full toolkit", "docs/cmap_heatmap.png"),
    ("Semantic network, full toolkit", "docs/cmap_semantic_network.png"),
]:
    print(f"{caption}  ({path})")
    display(Image(filename=path, width=620))

print("Source: CMAP Visualization Toolkit, BSD 3-Clause,")
print("https://github.com/Computational-Ethnography-Lab/cmap_visualization_toolkit")

## Lane 3. The large language model lane

The notebook does not call a model. It prints a prompt, you run it wherever you
already work, and you paste the answer back. No keys, no endpoints, no
configuration, and nothing leaves this machine on its own.

The prompt below follows a few rules that make the difference between output
you can use and output you have to clean up by hand:

- **CSV only, no prose.** A model that is allowed to explain itself will.
- **State the allowed values.** Without a closed list you get synonyms,
  invented categories, and inconsistent capitalisation.
- **Exactly one row out per row in.** Anything else silently misaligns your
  data.
- **Carry the identifiers through.** The prompt hands the model each row's
  `doc_id` and requires it back unchanged. That is what makes the answer
  checkable.
- **Post-process deterministically, outside the model.** Splitting, lowercasing
  and validating are code, not instructions.

That last pair matters more than it looks. A model with no access to the data
can return five perfectly formed, entirely invented rows. Parsing proves
syntax, not grounding, so the next cells check the returned identifiers
against the local source and reject any answer that disagrees.

In [ ]:
# Build the prompt. Copy everything between the rules into your assistant.

from cmap_demo.llm_handoff import PUBLIC_CSV_URL, build_llm_prompt

llm_rows = demo.head(5)
prompt = build_llm_prompt(llm_rows, n_rows=5, csv_url=PUBLIC_CSV_URL)

print("=" * 72)
print(prompt)
print("=" * 72)
print()
print(f"Expecting {len(llm_rows)} rows back, with these doc_ids: "
      f"{[int(i) for i in llm_rows['doc_id']]}")

In [ ]:
# Paste a real answer in below to check it, or run the deterministic stand-in
# so the notebook always produces a coded column.

from IPython.display import display

from cmap_demo.llm_handoff import (
    check_llm_response, code_by_stub, codes_to_source_shape, merge_code_columns,
)

# Paste the CSV an assistant returned between the triple quotes to check it.
# Leave it empty to skip. docs/llm_reference_output.csv holds one real answer.
llm_response = """"""

if llm_response.strip():
    checked = check_llm_response(llm_response, llm_rows)
    display(checked)
else:
    print("[--] No pasted response. Skipping the grounding check.")
    print("     Try it: paste an answer above and re-run this cell.")
print()

# The stand-in. Deterministic, inspectable, and clearly not a model result.
stub_codes = code_by_stub(demo)

# Merge the lanes into one column, in the same stringified-list shape the
# corpus uses. That shape is what lets the toolkit figures below read it.
merged = merge_code_columns(dictionary_codes, stub_codes)
demo["codes"] = codes_to_source_shape(merged)

coded_path = OUTPUT_DIR / "coded_demo.csv"
demo.to_csv(coded_path, index=False)
print(f"Wrote {coded_path}")

validate_cmap_frame(demo, csv_path=coded_path)
print()
demo[["doc_id", "word_count", "codes"]].head()

## Stage 4. Visualize

Two figures, both drawn from the codes this notebook just produced rather than
from anything shipped with the corpus. The code behind them is recycled
directly from the CMAP Visualization Toolkit, so what you see here is a
simplified version of the real thing rather than a lookalike.

These are the simple examples. The
[full toolkit](https://github.com/Computational-Ethnography-Lab/cmap_visualization_toolkit)
has the interactive versions: embeddings, clustering, t-SNE projections, and
semantic networks, along with its own Colab notebook.

In [ ]:
# Figure 1. Word cloud over the demo subset.

from cmap_demo.viz import generate_wordcloud

generate_wordcloud(
    demo["text"],
    title="Engineering oral histories",
    out_dir=OUTPUT_DIR,
)

In [ ]:
# Figure 2. How often the notebook's own codes appear on the same row.

from cmap_demo.viz import create_code_cooccurrence_heatmap

create_code_cooccurrence_heatmap(
    filepath=str(coded_path),
    num_codes=4,
    clustered=True,
    out_dir=OUTPUT_DIR,
)

## Where to go next

**The full toolkit.**
[CMAP Visualization Toolkit](https://github.com/Computational-Ethnography-Lab/cmap_visualization_toolkit)
(BSD 3-Clause) has the interactive versions of the figures above.
[CMAP QDPX Converter](https://github.com/Computational-Ethnography-Lab/cmap_qdpx_converter)
gets data out of ATLAS.ti, NVivo, or MAXQDA and into this format.

**Learning resources.**
The [AI wiki](https://github.com/Computational-Ethnography-Lab/ai-wiki) for
concepts and a glossary, and the
[teaching bibliography](https://github.com/Computational-Ethnography-Lab/teaching#v-bibliography)
for the curated reading list with DOIs. Both are kept current; this notebook
links them rather than duplicating them.

**Works cited here.**

- Abramson, Corey M., and Daniel Dohan. 2015. "Beyond Text: Using Arrays to
  Represent and Analyze Ethnographic Data." *Sociological Methodology*
  45(1):272–319. [doi:10.1177/0081175015578740](https://doi.org/10.1177/0081175015578740)
- Abramson, Corey M., Jacqueline Joslyn, Katharine A. Rendle, Sarah B. Garrett,
  and Daniel Dohan. 2018. "The Promises of Computational Ethnography."
  *Ethnography* 19(2):254–284.
  [doi:10.1177/1466138117725340](https://doi.org/10.1177/1466138117725340)
- Li, Zhuofan, Daniel Dohan, and Corey M. Abramson. 2021. "Qualitative Coding
  in the Computational Era." *Socius* 7.
  [doi:10.1177/23780231211062345](https://doi.org/10.1177/23780231211062345)
- Abramson, Corey M., Tara Prendergast, Zhuofan Li, and Daniel Dohan. 2026.
  "Qualitative Research in an Era of Artificial Intelligence." *Annual Review
  of Sociology* 52:20.1–20.27.
  [doi:10.1146/annurev-soc-011824-104836](https://doi.org/10.1146/annurev-soc-011824-104836)

The full list, including every repository and dataset used, is in
[`REFERENCES.md`](../REFERENCES.md).

**Licensing, once more.** The code is BSD 3-Clause. The dataset is not. It
keeps the IEEE History Center's restriction on quotation. The figures in the
model-lane cell belong to the CMAP Visualization Toolkit and are BSD 3-Clause.